In [1]:
import tensorflow as tf

In [2]:
import keras

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers

Using TensorFlow backend.


In [3]:
import numpy as np
import pandas as pd

from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
### 1.	Read the dataset

In [5]:
# import data set
df = pd.read_csv( 'bank.csv' )

In [6]:
df.shape

(10000, 14)

In [7]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
RowNumber          10000 non-null int64
CustomerId         10000 non-null int64
Surname            10000 non-null object
CreditScore        10000 non-null int64
Geography          10000 non-null object
Gender             10000 non-null object
Age                10000 non-null int64
Tenure             10000 non-null int64
Balance            10000 non-null float64
NumOfProducts      10000 non-null int64
HasCrCard          10000 non-null int64
IsActiveMember     10000 non-null int64
EstimatedSalary    10000 non-null float64
Exited             10000 non-null int64
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [9]:
# check how many missing values
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [10]:
### 	2. Drop the columns which are unique for all users like IDs

In [11]:
# drop ID column as it is useless for the model
if 'CustomerId' in df.columns:
    df.drop(['CustomerId','Surname'], axis=1, inplace = True)

In [12]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
RowNumber,10000.0,5000.500000,2886.895680,1.00,2500.75,5000.500,7500.2500,10000.00
CreditScore,10000.0,650.528800,96.653299,350.00,584.00,652.000,718.0000,850.00
Age,10000.0,38.921800,10.487806,18.00,32.00,37.000,44.0000,92.00
Tenure,10000.0,5.012800,2.892174,0.00,3.00,5.000,7.0000,10.00
Balance,10000.0,76485.889288,62397.405202,0.00,0.00,97198.540,127644.2400,250898.09
NumOfProducts,10000.0,1.530200,0.581654,1.00,1.00,1.000,2.0000,4.00
HasCrCard,10000.0,0.705500,0.455840,0.00,0.00,1.000,1.0000,1.00
IsActiveMember,10000.0,0.515100,0.499797,0.00,0.00,1.000,1.0000,1.00
EstimatedSalary,10000.0,100090.239881,57510.492818,11.58,51002.11,100193.915,149388.2475,199992.48
Exited,10000.0,0.203700,0.402769,0.00,0.00,0.000,0.0000,1.00


In [13]:
# Target Column Distribution
df.groupby(["Exited"]).count().T

Exited,0,1
RowNumber,7963,2037
CreditScore,7963,2037
Geography,7963,2037
Gender,7963,2037
Age,7963,2037
Tenure,7963,2037
Balance,7963,2037
NumOfProducts,7963,2037
HasCrCard,7963,2037
IsActiveMember,7963,2037


In [14]:
### 3.  Distinguish the feature and target set 

In [15]:
# Prepare X and Y
X = df.drop("Exited", axis=1)
y = df["Exited"]

In [16]:
# convert categorical data into numeric data - create separate columns for each category - through get_dummies
X_df = pd.get_dummies(X, drop_first = True)

In [17]:
X_df.shape

(10000, 12)

In [18]:
X_df.head(5)

,RowNumber,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,1,619,42,2,0.00,1,1,1,101348.88,0,0,0
1,2,608,41,1,83807.86,1,0,1,112542.58,0,1,0
2,3,502,42,8,159660.80,3,1,0,113931.57,0,0,0
3,4,699,39,1,0.00,2,0,0,93826.63,0,0,0
4,5,850,43,2,125510.82,1,1,1,79084.10,0,1,0


In [19]:
# encode class values (Y) as integers for binary classification
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(y)
encoded_y = encoder.transform(y)

In [20]:
# scale the input features
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_df = sc.fit_transform(X_df)

In [21]:
### 4.  Divide the data set into Train and test sets

In [22]:
from sklearn.model_selection import train_test_split

# Split the data into training and test set in the ratio of 70:30 respectively 
X_train, X_test, y_train, y_test = train_test_split(X_df, encoded_y, test_size = 0.3, random_state = 7)

In [23]:
### 5.  Normalize the train and test data 

In [24]:
from sklearn.preprocessing import Normalizer

transformer = Normalizer().fit(X_train)
X_train = transformer.transform(X_train)
X_test = transformer.transform(X_test)

In [25]:
### 6.  Initialize & build the model 

In [26]:
from keras.models import Sequential
from keras.layers import Dense

In [27]:
# create model
model_init = tf.keras.models.Sequential()

model_init.add(tf.keras.layers.Dense(12,input_dim=12,activation='relu'))
model_init.add(tf.keras.layers.Dense(8, activation='relu'))
model_init.add(tf.keras.layers.Dense(1, activation='sigmoid'))

# Compile model
model_init.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [28]:
# Fit the model
model_init.fit(X_train, y_train, epochs=10, batch_size=10)

W0818 20:26:40.778074 13300 deprecation.py:323] From C:\Users\Suchi\Anaconda3\lib\site-packages\tensorflow\python\ops\math_grad.py:1250: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Train on 7000 samples
Epoch 1/10
7000/7000 [==============================] - 1s 99us/sample - loss: 0.4992 - accuracy: 0.7849
Epoch 2/10
7000/7000 [==============================] - 1s 91us/sample - loss: 0.4214 - accuracy: 0.8159
Epoch 3/10
7000/7000 [==============================] - 1s 89us/sample - loss: 0.4077 - accuracy: 0.8254
Epoch 4/10
7000/7000 [==============================] - 1s 87us/sample - loss: 0.3947 - accuracy: 0.8307
Epoch 5/10
7000/7000 [==============================] - 1s 88us/sample - loss: 0.3795 - accuracy: 0.8446
Epoch 6/10
7000/7000 [==============================] - 1s 90us/sample - loss: 0.3671 - accuracy: 0.8486
Epoch 7/10
7000/7000 [==============================] - 1s 89us/sample - loss: 0.3580 - accuracy: 0.8529
Epoch 8/10
7000/7000 [==============================] - 1s 89us/sample - loss: 0.3527 - accuracy: 0.8550
Epoch 9/10
7000/7000 [==============================] - 1s 89us/sample - loss: 0.3501 - accuracy: 0.8586
Epoch 10/10
7000/7000 [==========

In [29]:
# evaluate the model
score = model_init.evaluate(X_test, y_test, verbose=0)
# Print test accuracy
print('\n', 'Test accuracy for Initial Model :', score[1]*100)


 Test accuracy for Initial Model : 85.26666760444641


In [30]:
#### 8.  Optimize the model

In [31]:
# Function to create model
def create_model(optimizer,activation):
    # create model
    model = tf.keras.models.Sequential()

    model.add(tf.keras.layers.Dense(12,activation=activation))
    model.add(tf.keras.layers.Dense(8, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    
    # SGD with momentum and decay 
    #sgd = tf.optimizers.SGD(lr=learn_rate, decay=1e-6, momentum=0.9)
    
    # Compile model
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model


In [32]:
# create model using KerasClassifier
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier

model = KerasClassifier(build_fn=create_model, epochs=10, batch_size=10, verbose=0)

In [33]:
# Use scikit-learn to grid search the hyperparameters
from sklearn.model_selection import GridSearchCV

# define the grid search parameters

optimizers = ['sgd', 'adam', 'adamax']
activation = ['relu', 'tanh', 'sigmoid']
#learn_rate = [0.001, 0.01, 0.1, 0.2, 0.3]
param_grid = dict(optimizer=optimizers,activation=activation)

# GridSearchCV to get optimized parameters
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1)
grid_result = grid.fit(X_train, y_train)

C:\Users\Suchi\Anaconda3\lib\site-packages\sklearn\model_selection\_split.py:1978: FutureWarning: The default value of cv will change from 3 to 5 in version 0.22. Specify it explicitly to silence this warning.
  warnings.warn(CV_WARNING, FutureWarning)


In [34]:
# summarize results from GridSearchCV

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.841714 using {'activation': 'relu', 'optimizer': 'adam'}
0.800429 (0.007913) with: {'activation': 'relu', 'optimizer': 'sgd'}
0.841714 (0.010527) with: {'activation': 'relu', 'optimizer': 'adam'}
0.813571 (0.001231) with: {'activation': 'relu', 'optimizer': 'adamax'}
0.809000 (0.002592) with: {'activation': 'tanh', 'optimizer': 'sgd'}
0.836429 (0.014431) with: {'activation': 'tanh', 'optimizer': 'adam'}
0.818143 (0.003606) with: {'activation': 'tanh', 'optimizer': 'adamax'}
0.795429 (0.001435) with: {'activation': 'sigmoid', 'optimizer': 'sgd'}
0.812286 (0.004735) with: {'activation': 'sigmoid', 'optimizer': 'adam'}
0.803714 (0.004068) with: {'activation': 'sigmoid', 'optimizer': 'adamax'}


In [35]:
# Check the accuracy of best model
bestmodel = grid.best_estimator_.model
# evaluate the model
score = bestmodel.evaluate(X_test, y_test, verbose=0)
# Print test accuracy
print('\n', 'Test accuracy for Optimized Model :', score[1]*100)


 Test accuracy for Optimized Model : 85.6333315372467


In [36]:
y_pred = bestmodel.predict(X_test)
print(y_pred)

[[0.61934805]
 [0.08712575]
 [0.01819882]
 ...
 [0.00424525]
 [0.02530956]
 [0.22119996]]


In [37]:
### 9. Predict the results using 0.5 as a threshold

In [38]:
# Predict Class 
class_pred = []
for val in y_pred:
    if val> 0.5: # use Threshold of 0.5
        class_pred.append(1)
    else:
        class_pred.append(0)
print(class_pred)

[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [39]:
from sklearn.metrics import confusion_matrix, accuracy_score

In [40]:
print('\n', 'Confusion Matrix of Optimized Model :')
confusion_matrix(y_test,class_pred)


 Confusion Matrix of Optimized Model :


array([[2314,   81],
       [ 350,  255]], dtype=int64)

In [41]:
# Print accuracy score
print('\n', 'Accuracy Score of Optimized Model :', accuracy_score(y_test,class_pred)*100)


 Accuracy Score of Optimized Model : 85.63333333333333
